# Vision Park — remote PKLot pilot preparation and EfficientNet-B0 training
Run in **Google Colab** from any computer or tablet with a browser (no Mac needed).

This notebook:
1. Connects your Google Drive.
2. Uses the existing **PKLot.tar.gz** (4.56 GiB) without extracting the 19+ GiB archive.
3. Generates real 224×224 cropped images in `train.npz`, `validation.npz`, `test.npz`.
4. Fine-tunes EfficientNet-B0 and saves genuine evaluation results to Drive.

**Important:** This notebook does not start itself. You must authorize Google Drive access and run the cells in your own Colab session. GPU access depends on Colab availability. The camera/miniature-toy-car dataset is a separate step.

In [ ]:
# 1 — Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 2 — Fetch the project's preparation and training code
import os, subprocess, sys
from pathlib import Path

REPO = Path("/content/vp")
if not (REPO / ".git").exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--single-branch",
         "--branch", "feat/visionpark-audit-foundations",
         "https://github.com/nidaa1308-arch/vp.git", str(REPO)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     str(REPO / "ai/requirements.txt")],
    check=True,
)
print("Project ready:", REPO)

In [ ]:
# 3 — Find the EXISTING archive, without downloading it again if possible
from pathlib import Path
import subprocess, sys
ARCHIVE = Path("/content/drive/MyDrive/VisionPark_AI/Dataset/PKLot.tar.gz")
SHARED_FILE_ID = "1p1Y5psXQnhTht4p3wYnUVjAoIaXclg-4"

if not ARCHIVE.is_file():
    print("The default Drive location was not found.")
    print("Attempting to download the Drive shared link into Colab temporary storage.")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
    ARCHIVE = Path("/content/PKLot.tar.gz")
    try:
        subprocess.run(
            ["gdown", "--id", SHARED_FILE_ID, "-O", str(ARCHIVE)],
            check=True,
        )
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            "Colab could not access the shared 4.56 GiB file. "
            "Sign into the account that owns the archive and place it at "
            "MyDrive/VisionPark_AI/Dataset/PKLot.tar.gz. "
            "Then rerun this cell. Google may require explicit sharing access."
        ) from exc
assert ARCHIVE.is_file() and ARCHIVE.stat().st_size > 4_000_000_000, (
    f"Expected PKLot.tar.gz about 4.56 GiB, found {ARCHIVE}"
)
print("Archive ready:", ARCHIVE, "bytes:", ARCHIVE.stat().st_size)

In [ ]:
# 4 — Generate the three small pilot files (no full PKLot extraction)
from pathlib import Path
DATA_ROOT = Path("/content/drive/MyDrive/VisionPark_AI")
PILOT = DATA_ROOT / "Processed_Dataset" / "Pilot"
os.environ["VISIONPARK_DATA_DIR"] = str(DATA_ROOT)
parts = [PILOT / f"{name}.npz" for name in ("train", "validation", "test")]

if all(p.is_file() for p in parts):
    print("All three pilot files already exist; not overwriting them.")
elif any(p.is_file() for p in parts):
    raise RuntimeError(
        "Only some pilot files exist. Back them up or finish recovering "
        "the missing files before regenerating the set."
    )
else:
    subprocess.run([
        sys.executable, str(REPO / "ai/prepare_pilot_from_archive.py"),
        "--archive", str(ARCHIVE),
        "--output-dir", str(PILOT),
        "--split-dir", str(DATA_ROOT / "Processed_Dataset"),
        "--samples-per-class", "1000",
    ], check=True)
for p in parts:
    assert p.is_file(), f"Missing output: {p}"
    print(p.name, f"{p.stat().st_size / 1024**2:.1f} MiB")

In [ ]:
# 5 — Verify that datasets contain REAL RGB crops and both labels
import numpy as np
for name in ("train", "validation", "test"):
    with np.load(PILOT / f"{name}.npz", allow_pickle=False) as dataset:
        assert {"X", "y"} <= set(dataset.files), f"{name}: missing X or y"
        x, y = dataset["X"], dataset["y"]
        assert x.ndim == 4 and x.shape[1:] == (224, 224, 3)
        assert x.dtype == np.uint8 and set(np.unique(y).tolist()) == {0, 1}
        print(name, "shape", x.shape, "labels (vacant, occupied):",
              dict(zip(*np.unique(y, return_counts=True))))
print("Saved datasets are ready for training.")

In [ ]:
# 6 — Fine-tune pretrained EfficientNet-B0 (5 head + 5 fine-tune epochs)
# Optional: Runtime -> Change runtime type -> T4 GPU.
import torch
print("CUDA available:", torch.cuda.is_available())
subprocess.run(
    [sys.executable, str(REPO / "ai/finetune_efficientnet_local.py")],
    check=True,
    env=os.environ.copy(),
)
print("Checkpoint saved in:", DATA_ROOT / "Trained_Models" / "EfficientNet_B0_Pilot_local")

In [ ]:
# 7 — Reload the checkpoint and independently predict actual test images
MODEL = DATA_ROOT / "Trained_Models/EfficientNet_B0_Pilot_local/VisionPark_EfficientNet_B0_Pilot_v2.pt"
OUTPUT = DATA_ROOT / "Results/EfficientNet_B0_Pilot_local/verified"
subprocess.run([
    sys.executable, str(REPO / "ai/evaluate_checkpoint.py"),
    "--checkpoint", str(MODEL),
    "--test", str(PILOT / "test.npz"),
    "--output", str(OUTPUT),
], check=True, env=os.environ.copy())
print("Evaluation saved:", OUTPUT)
print((OUTPUT / "metrics.json").read_text())

## Interpreting results

Check `MyDrive/VisionPark_AI/Results/EfficientNet_B0_Pilot_local/verified/metrics.json`.
The file contains actual accuracy, precision, recall, F1 and a confusion matrix measured on cropped test images—not pre-recorded predictions.

This is a **PKLot real-car baseline**. It does not guarantee performance on your **ESP32-CAM miniature toy-car setup**. Collect and evaluate new miniature parking images later.

If the preparation cell reports no valid images or an unexpected archive layout, copy the traceback into ChatGPT so we can correct it rather than claiming success.